In [4]:
# EXTRAÇÃO DE CARACTERÍSTICAS (AVANÇADAS E GLOBAIS)

import os
import glob
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
from scipy.stats import entropy
import warnings

# Ignorar warnings de divisão por zero
warnings.filterwarnings('ignore')

print("Notebook 3: Feature engineering")
print("="*80)

PASTA_CSV = '/workspaces/EyeTracking/data/csv/'
PASTA_PROCESSED = '/workspaces/EyeTracking/data/processed/'
os.makedirs(PASTA_PROCESSED, exist_ok=True)

arquivos = glob.glob(os.path.join(PASTA_CSV, '*.csv'))
lista_features = []

# Lista das colunas de AOI (criadas no notebook 2)
aois = [
    'AOI_Olho_Esquerdo', 'AOI_Olho_Direito', 'AOI_Ambos_Olhos', 
    'AOI_Nariz', 'AOI_Boca', 'AOI_Triangulo', 'AOI_Rosto', 'AOI_Fora'
]

print(f"Iniciando extração de features para {len(arquivos)} pacientes...\n")

for caminho in arquivos:
    nome_puro = os.path.basename(caminho).replace('.csv', '')
    grupo = "TEA" if "TEA" in nome_puro.upper() else "CONTROLE"
    
    try:
        df = pd.read_csv(caminho)
        
        # HIGIENIZAÇÃO DE TIPOS: Corrige vírgula decimal e força conversão numérica
        colunas_numericas = ['Gaze_X', 'Gaze_Y', 'Pupil_L', 'Pupil_R']
        for col in colunas_numericas:
            if col in df.columns:
                if df[col].dtype == object or df[col].dtype == str:
                    df[col] = df[col].astype(str).str.replace(',', '.')
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # Cria a pupila média de forma segura
        colunas_pupila = [col for col in ['Pupil_L', 'Pupil_R'] if col in df.columns]
        if colunas_pupila:
            df['Pupila_Media'] = df[colunas_pupila].mean(axis=1)
        else:
            df['Pupila_Media'] = np.nan
        
        # Filtros principais
        df_face = df[df['Fase_Estimulo'] == 'Exposicao_Face'].copy()
        df_janela = df[df['Fase_Estimulo'] == 'Janela_Baseline'].copy()
        
        # Dicionário que guarda todas as features do paciente
        paciente_ft = {
            'Paciente': nome_puro,
            'Grupo': grupo
        }
        
        # 1. TAXA DE FUGA VISUAL (data loss comportamental)
        fuga_global = df_face['Gaze_X'].isnull().mean() * 100 if len(df_face) > 0 else 100
        paciente_ft['Fuga_Visual_Global_%'] = fuga_global
        
        # 2. TEMPO DE RETENÇÃO (dwell time) - função auxiliar
        for aoi in aois:
            if aoi in df_face.columns:
                df_face[aoi] = df_face[aoi].fillna(False).astype(bool)
            else:
                df_face[aoi] = False
            
        def calcular_retencao(df_recorte, prefixo):
            if len(df_recorte) == 0:
                for aoi in aois: 
                    paciente_ft[f"{prefixo}_{aoi}_%"] = 0.0
                return
            for aoi in aois:
                paciente_ft[f"{prefixo}_{aoi}_%"] = df_recorte[aoi].mean() * 100.0

        # Cálculos de retenção
        calcular_retencao(df_face, "Global")
        calcular_retencao(df_face[df_face['Tipo_Estimulo'] == 'Humano'], "Humano")
        calcular_retencao(df_face[df_face['Tipo_Estimulo'] == 'Desenho'], "Desenho")
        
        for emocao in ['Feliz', 'Neutro', 'Raiva']:
            calcular_retencao(df_face[(df_face['Tipo_Estimulo'] == 'Humano') & (df_face['Emocao'] == emocao)], f"Humano_{emocao}")
        
        # Contraste humano vs desenho (olhos)
        paciente_ft['Contraste_Olhos_Humano_vs_Desenho'] = paciente_ft.get('Humano_AOI_Ambos_Olhos_%', 0) - paciente_ft.get('Desenho_AOI_Ambos_Olhos_%', 0)

        # 3. ÍNDICES CLÍNICOS
        boca_global = paciente_ft.get('Global_AOI_Boca_%', 0)
        paciente_ft['EMI_Global'] = paciente_ft.get('Global_AOI_Ambos_Olhos_%', 0) / boca_global if boca_global > 0 else 0
        
        fora_global = paciente_ft.get('Global_AOI_Fora_%', 0)
        paciente_ft['Indice_Social_Global'] = paciente_ft.get('Global_AOI_Triangulo_%', 0) / fora_global if fora_global > 0 else 0

        # 4. MAPEAMENTO DE EXPLORAÇÃO (velocidade e TTFF)
        df_face['Distancia_X'] = df_face['Gaze_X'].diff().abs()
        df_face['Distancia_Y'] = df_face['Gaze_Y'].diff().abs()
        df_face['Velocidade_Sacadica'] = np.sqrt(df_face['Distancia_X']**2 + df_face['Distancia_Y']**2)
        paciente_ft['Velocidade_Sacadica_Media'] = df_face['Velocidade_Sacadica'].mean() if len(df_face) > 0 else 0

        # TTFF (time to first fixation) para os olhos
        ttff_lista = []
        for face_id in df_face['Face_ID'].unique():
            df_bloco = df_face[df_face['Face_ID'] == face_id]
            inicio_bloco = df_bloco['Time_ms'].min()
            
            fixacoes_olho = df_bloco[df_bloco['AOI_Ambos_Olhos'] == True]
            if not fixacoes_olho.empty:
                primeiro_olhar = fixacoes_olho['Time_ms'].iloc[0]
                ttff = primeiro_olhar - inicio_bloco
                ttff_lista.append(ttff)
                
        paciente_ft['TTFF_Medio_Olhos_ms'] = np.mean(ttff_lista) if len(ttff_lista) > 0 else 2700

        # 5. DINÂMICA DE FIXAÇÕES
        df_face['Is_Fixation'] = df_face['Velocidade_Sacadica'] < 2.0
        df_face['Fixation_Block'] = (df_face['Is_Fixation'] != df_face['Is_Fixation'].shift()).cumsum()
        fixations = df_face[df_face['Is_Fixation'] == True].groupby('Fixation_Block').size()
        
        paciente_ft['Num_Total_Fixacoes'] = len(fixations)
        paciente_ft['Duracao_Media_Fixacao_ms'] = fixations.mean() if len(fixations) > 0 else 0

        # 6. DISPERSÃO ESPACIAL (Convex Hull)
        pontos_validos = df_face[['Gaze_X', 'Gaze_Y']].dropna()
        if len(pontos_validos.drop_duplicates()) > 3:
            hull = ConvexHull(pontos_validos)
            paciente_ft['Area_Dispersao_ConvexHull'] = hull.volume
        else:
            paciente_ft['Area_Dispersao_ConvexHull'] = 0

        # 7. ENTROPIA DE SHANNON
        probabilidades = [df_face[aoi].mean() for aoi in aois if aoi in df_face.columns]
        probabilidades = [p for p in probabilidades if p > 0]
        paciente_ft['Entropia_Shannon_Olhar'] = entropy(probabilidades) if len(probabilidades) > 0 else 0

        # 8. MATRIZ DE TRANSIÇÃO
        df_face['Transicao_Olho_Boca'] = (df_face['AOI_Ambos_Olhos'].shift(1) == True) & (df_face['AOI_Boca'] == True)
        paciente_ft['Num_Transicoes_Olho_Boca'] = df_face['Transicao_Olho_Boca'].sum()

        # 9. REATIVIDADE PUPILAR
        baseline_pupila = df_janela['Pupila_Media'].mean() if len(df_janela) > 0 else np.nan
        face_pupila = df_face['Pupila_Media'].mean() if len(df_face) > 0 else np.nan
        
        paciente_ft['Pupila_Baseline'] = baseline_pupila if not np.isnan(baseline_pupila) else 0
        paciente_ft['Pupila_Face_Global'] = face_pupila if not np.isnan(face_pupila) else 0
        paciente_ft['Pupila_Reatividade_Global'] = (face_pupila - baseline_pupila) if (not np.isnan(face_pupila) and not np.isnan(baseline_pupila)) else 0
        
        lista_features.append(paciente_ft)

    except Exception as e:
        print(f"Erro ao extrair features de '{nome_puro}': {e}")

# CONSOLIDAÇÃO FINAL
df_features = pd.DataFrame(lista_features)
df_features = df_features.fillna(0)

CAMINHO_MATRIZ = os.path.join(PASTA_PROCESSED, 'matriz_features_ml.csv')
df_features.to_csv(CAMINHO_MATRIZ, index=False)

print("\n" + "="*80)
print(f"EXTRAÇÃO CONCLUÍDA COM SUCESSO PARA OS {len(df_features)} PACIENTES!")
print(f"Matriz consolidada salva em: {CAMINHO_MATRIZ}")

Notebook 3: Feature engineering
Iniciando extração de features para 41 pacientes...


EXTRAÇÃO CONCLUÍDA COM SUCESSO PARA OS 41 PACIENTES!
Matriz consolidada salva em: /workspaces/EyeTracking/data/processed/matriz_features_ml.csv


In [5]:
# EXTRAÇÃO DE CARACTERÍSTICAS (BCEA)

import os
import glob
import numpy as np
import pandas as pd

def calcular_bcea(gx, gy):
    """
    Calcula a Bivariate Contour Ellipse Area (BCEA) para coordenadas de olhar (X, Y).
    Filtra pontos inválidos (zeros e NaNs).
    """
    mask = ~np.isnan(gx) & ~np.isnan(gy) & (gx != 0) & (gy != 0)
    gx_clean, gy_clean = gx[mask], gy[mask]
    
    if len(gx_clean) < 3:
        return 0.0
    
    sx = np.std(gx_clean, ddof=1)
    sy = np.std(gy_clean, ddof=1)
    
    if sx == 0 or sy == 0:
        return 0.0
    
    rho = np.corrcoef(gx_clean, gy_clean)[0, 1]
    if np.isnan(rho):
        rho = 0.0
        
    rho = np.clip(rho, -0.999, 0.999)
    area = 2 * np.pi * sx * sy * np.sqrt(max(0.0, 1 - rho**2))
    return float(area)

def processar_arquivo_sujeito(caminho_csv):
    df = pd.read_csv(caminho_csv)
    
    # Validação de dados válidos de rastreamento
    gaze_valido = df[(df['Gaze_X'] != 0) & (df['Gaze_Y'] != 0)]
    if len(gaze_valido) < 50:
        return None
    
    nome_arquivo = os.path.basename(caminho_csv).replace('.csv', '')
    id_sujeito = nome_arquivo.split()[0]  # Extrai identificador (ex: 'T28')
    
    # Define o rótulo de classe (Target: 1 para TEA, 0 para Controle/TDC) com base no nome do arquivo
    nome_upper = nome_arquivo.upper()
    if 'TEA' in nome_upper:
        target = 1
    elif any(termo in nome_upper for termo in ['TDC', 'CTL', 'CONTROLE', 'TD', 'TYPICAL']):
        target = 0
    else:
        target = np.nan

    # 1. BCEA Geral
    bcea_geral = calcular_bcea(df['Gaze_X'].values, df['Gaze_Y'].values)
    
    # 2. BCEA por Emoção (Usando a coluna 'Emocao')
    bcea_emocoes = {}
    if 'Emocao' in df.columns:
        for emo in ['feliz', 'neutro', 'raiva']:
            sub_df = df[df['Emocao'].astype(str).str.lower() == emo]
            bcea_emocoes[f'BCEA_{emo.capitalize()}'] = calcular_bcea(sub_df['Gaze_X'].values, sub_df['Gaze_Y'].values)
    else:
        for emo in ['Feliz', 'Neutro', 'Raiva']:
            bcea_emocoes[f'BCEA_{emo}'] = 0.0

    # 3. Consolidação de Features Oculomotoras e Proporções em AOIs
    features = {
        'ID_Sujeito': id_sujeito,
        'Nome_Arquivo': nome_arquivo,
        'Target': target,
        'BCEA_Geral': bcea_geral,
        'BCEA_Feliz': bcea_emocoes.get('BCEA_Feliz', 0.0),
        'BCEA_Neutro': bcea_emocoes.get('BCEA_Neutro', 0.0),
        'BCEA_Raiva': bcea_emocoes.get('BCEA_Raiva', 0.0),
        'Gaze_X_Mean': gaze_valido['Gaze_X'].mean(),
        'Gaze_Y_Mean': gaze_valido['Gaze_Y'].mean(),
        'Gaze_X_Median': gaze_valido['Gaze_X'].median(),
        'Gaze_Y_Median': gaze_valido['Gaze_Y'].median(),
        'Gaze_X_Std': gaze_valido['Gaze_X'].std(),
        'Gaze_Y_Std': gaze_valido['Gaze_Y'].std(),
    }

    # Proporção de fixação nas AOIs (áreas de interesse)
    aoi_cols = ['AOI_Ambos_Olhos', 'AOI_Nariz', 'AOI_Boca', 'AOI_Triangulo', 'AOI_Rosto', 'AOI_Fora']
    total_linhas = len(df)
    for col in aoi_cols:
        if col in df.columns:
            features[f'Prop_{col}'] = df[col].sum() / total_linhas if total_linhas > 0 else 0.0

    return features

# --- Execução da Extração ---
diretorio_csvs = "/workspaces/EyeTracking/data/csv/*.csv" 
lista_arquivos = glob.glob(diretorio_csvs)

dados_processados = []
for file_path in lista_arquivos:
    res = processar_arquivo_sujeito(file_path)
    if res is not None:
        dados_processados.append(res)

df_mega = pd.DataFrame(dados_processados)

# Exibir resumo do processamento
print("=" * 60)
print(f"Sujeitos processados com sucesso: {len(df_mega)} de {len(lista_arquivos)}")
print("\nDistribuição das classes extraídas (1 = TEA, 0 = Controle):")
print(df_mega['Target'].value_counts(dropna=False))
print("\nPrimeiras 5 linhas da tabela consolidada:")
print(df_mega[['ID_Sujeito', 'Target', 'BCEA_Geral', 'BCEA_Feliz', 'BCEA_Neutro', 'BCEA_Raiva']].head())
print("=" * 60)

# Salvar arquivo na pasta de dados do projeto
caminho_saida = "/workspaces/EyeTracking/data/df_mega_bcea.csv"
df_mega.to_csv(caminho_saida, index=False)
print(f"Arquivo consolidado salvo com sucesso em:\n{caminho_saida}")

Sujeitos processados com sucesso: 41 de 41

Distribuição das classes extraídas (1 = TEA, 0 = Controle):
Target
0    23
1    18
Name: count, dtype: int64

Primeiras 5 linhas da tabela consolidada:
  ID_Sujeito  Target     BCEA_Geral     BCEA_Feliz    BCEA_Neutro  \
0        T28       1  397693.543388  423432.174400  373675.165885   
1        T26       1  103624.416612  121065.679692   61017.150153   
2        T41       0  193339.069092  199413.084604  143583.612282   
3        T30       1  159429.076966  158865.009059  162445.573419   
4        T54       0   50390.217638   45889.130060   46865.236586   

      BCEA_Raiva  
0  365877.520761  
1  108642.172068  
2  200828.070505  
3  139207.431329  
4   54098.708121  
Arquivo consolidado salvo com sucesso em:
/workspaces/EyeTracking/data/df_mega_bcea.csv


In [7]:
import os
import glob
import numpy as np
import pandas as pd

def calcular_bcea(gx, gy):
    """Calcula a Bivariate Contour Ellipse Area (BCEA)."""
    mask = ~np.isnan(gx) & ~np.isnan(gy) & (gx != 0) & (gy != 0)
    gx_clean, gy_clean = gx[mask], gy[mask]
    if len(gx_clean) < 3:
        return 0.0
    sx, sy = np.std(gx_clean, ddof=1), np.std(gy_clean, ddof=1)
    if sx == 0 or sy == 0:
        return 0.0
    rho = np.corrcoef(gx_clean, gy_clean)[0, 1]
    rho = 0.0 if np.isnan(rho) else np.clip(rho, -0.999, 0.999)
    return float(2 * np.pi * sx * sy * np.sqrt(max(0.0, 1 - rho**2)))

def calcular_entropia_scanpath(df):
    """Calcula a entropia de Shannon das transições visuais entre AOIs."""
    aoi_cols = [c for c in ['AOI_Ambos_Olhos', 'AOI_Nariz', 'AOI_Boca', 'AOI_Fora'] if c in df.columns]
    if not aoi_cols:
        return 0.0
    
    # Garantir conversão numérica das AOIs
    for col in aoi_cols:
        if df[col].dtype == object or str(df[col].dtype).startswith('string'):
            df[col] = df[col].astype(str).str.replace(',', '.')
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        
    aoi_ativa = df[aoi_cols].idxmax(axis=1)
    counts = aoi_ativa.value_counts(normalize=True).values
    probs = counts[counts > 0]
    return float(-np.sum(probs * np.log2(probs)))

def processar_arquivo_completo(caminho_csv):
    df = pd.read_csv(caminho_csv)
    
    # 1. Tratamento e Coerção Numérica de Colunas Sensíveis
    cols_numericas = ['Gaze_X', 'Gaze_Y', 'Pupil_L', 'Pupil_R']
    for col in cols_numericas:
        if col in df.columns:
            if df[col].dtype == object or str(df[col].dtype).startswith('string'):
                df[col] = df[col].astype(str).str.replace(',', '.')
            df[col] = pd.to_numeric(df[col], errors='coerce')

    aoi_cols_todas = ['AOI_Ambos_Olhos', 'AOI_Nariz', 'AOI_Boca', 'AOI_Triangulo', 'AOI_Rosto', 'AOI_Fora']
    for col in aoi_cols_todas:
        if col in df.columns:
            if df[col].dtype == object or str(df[col].dtype).startswith('string'):
                df[col] = df[col].astype(str).str.replace(',', '.')
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # Filtragem de registros válidos de olhar
    gaze_valido = df[(df['Gaze_X'].fillna(0) != 0) & (df['Gaze_Y'].fillna(0) != 0)]
    if len(gaze_valido) < 50:
        return None
    
    nome_arquivo = os.path.basename(caminho_csv).replace('.csv', '')
    id_sujeito = nome_arquivo.split()[0]
    
    nome_upper = nome_arquivo.upper()
    target = 1 if 'TEA' in nome_upper else (0 if any(t in nome_upper for t in ['TDC', 'CTL', 'CONTROLE', 'TD']) else np.nan)

    # 2. BCEA Geral e por Emoção
    bcea_geral = calcular_bcea(df['Gaze_X'].values, df['Gaze_Y'].values)
    bcea_emo = {}
    if 'Emocao' in df.columns:
        for emo in ['feliz', 'neutro', 'raiva']:
            sub = df[df['Emocao'].astype(str).str.lower() == emo]
            bcea_emo[emo] = calcular_bcea(sub['Gaze_X'].values, sub['Gaze_Y'].values)
    else:
        bcea_emo = {'feliz': 0.0, 'neutro': 0.0, 'raiva': 0.0}

    # 3. Proporções de AOI e Razões
    tot = len(df)
    prop_olhos = df['AOI_Ambos_Olhos'].sum() / tot if 'AOI_Ambos_Olhos' in df.columns else 0.0
    prop_boca = df['AOI_Boca'].sum() / tot if 'AOI_Boca' in df.columns else 0.0
    prop_nariz = df['AOI_Nariz'].sum() / tot if 'AOI_Nariz' in df.columns else 0.0
    prop_rosto = df['AOI_Rosto'].sum() / tot if 'AOI_Rosto' in df.columns else 0.0
    prop_fora = df['AOI_Fora'].sum() / tot if 'AOI_Fora' in df.columns else 0.0

    razao_olhos_boca = prop_olhos / (prop_boca + 1e-5)

    # 4. Deltas de Reatividade Emocional
    delta_bcea_raiva_neutro = bcea_emo['raiva'] - bcea_emo['neutro']
    delta_bcea_feliz_neutro = bcea_emo['feliz'] - bcea_emo['neutro']

    # 5. Média e Desvio de Pupila (tratando eventuais NaNs)
    pupil_mean_l = df['Pupil_L'].mean() if 'Pupil_L' in df.columns else np.nan
    pupil_mean_r = df['Pupil_R'].mean() if 'Pupil_R' in df.columns else np.nan
    pupil_std_l = df['Pupil_L'].std() if 'Pupil_L' in df.columns else np.nan
    pupil_std_r = df['Pupil_R'].std() if 'Pupil_R' in df.columns else np.nan

    pupil_mean = np.nanmean([pupil_mean_l, pupil_mean_r])
    pupil_std = np.nanmean([pupil_std_l, pupil_std_r])

    features = {
        'ID_Sujeito': id_sujeito,
        'Target': target,
        'BCEA_Geral': bcea_geral,
        'BCEA_Feliz': bcea_emo['feliz'],
        'BCEA_Neutro': bcea_emo['neutro'],
        'BCEA_Raiva': bcea_emo['raiva'],
        'Delta_BCEA_Raiva_Neutro': delta_bcea_raiva_neutro,
        'Delta_BCEA_Feliz_Neutro': delta_bcea_feliz_neutro,
        'Prop_Olhos': prop_olhos,
        'Prop_Boca': prop_boca,
        'Prop_Nariz': prop_nariz,
        'Prop_Rosto': prop_rosto,
        'Prop_Fora': prop_fora,
        'Razao_Olhos_Boca': razao_olhos_boca,
        'Entropia_Scanpath': calcular_entropia_scanpath(df),
        'Gaze_X_Std': gaze_valido['Gaze_X'].std(),
        'Gaze_Y_Std': gaze_valido['Gaze_Y'].std(),
        'Pupil_Mean': 0.0 if np.isnan(pupil_mean) else float(pupil_mean),
        'Pupil_Std': 0.0 if np.isnan(pupil_std) else float(pupil_std)
    }
    return features

# --- Execução da Extração ---
lista_arquivos = glob.glob("/workspaces/EyeTracking/data/csv/*.csv")
dados = [res for f in lista_arquivos if (res := processar_arquivo_completo(f)) is not None]

df_mega = pd.DataFrame(dados)

# Salvar arquivo de saída
caminho_saida = "/workspaces/EyeTracking/data/df_mega_bcea.csv"
df_mega.to_csv(caminho_saida, index=False)

print("=" * 60)
print(f"Processamento concluído com sucesso!")
print(f"Total de sujeitos validados: {len(df_mega)}")
print(f"Total de atributos extraídos: {df_mega.shape[1]}")
print(f"Arquivo salvo em: {caminho_saida}")
print("=" * 60)

Processamento concluído com sucesso!
Total de sujeitos validados: 41
Total de atributos extraídos: 19
Arquivo salvo em: /workspaces/EyeTracking/data/df_mega_bcea.csv


In [8]:
import os
import glob
import numpy as np
import pandas as pd

def calcular_bcea(gx, gy):
    mask = ~np.isnan(gx) & ~np.isnan(gy) & (gx != 0) & (gy != 0)
    gx_clean, gy_clean = gx[mask], gy[mask]
    if len(gx_clean) < 3:
        return 0.0
    sx, sy = np.std(gx_clean, ddof=1), np.std(gy_clean, ddof=1)
    if sx == 0 or sy == 0:
        return 0.0
    rho = np.corrcoef(gx_clean, gy_clean)[0, 1]
    rho = 0.0 if np.isnan(rho) else np.clip(rho, -0.999, 0.999)
    return float(2 * np.pi * sx * sy * np.sqrt(max(0.0, 1 - rho**2)))

def processar_arquivo_completo(caminho_csv):
    df = pd.read_csv(caminho_csv)
    
    # Tratamento numérico
    for col in ['Gaze_X', 'Gaze_Y', 'Pupil_L', 'Pupil_R', 'AOI_Ambos_Olhos', 'AOI_Nariz', 'AOI_Boca', 'AOI_Rosto', 'AOI_Fora']:
        if col in df.columns:
            if df[col].dtype == object or str(df[col].dtype).startswith('string'):
                df[col] = df[col].astype(str).str.replace(',', '.')
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    gaze_valido = df[(df['Gaze_X'] != 0) & (df['Gaze_Y'] != 0)].copy()
    if len(gaze_valido) < 50:
        return None
    
    nome_arquivo = os.path.basename(caminho_csv).replace('.csv', '')
    id_sujeito = nome_arquivo.split()[0]
    nome_upper = nome_arquivo.upper()
    target = 1 if 'TEA' in nome_upper else (0 if any(t in nome_upper for t in ['TDC', 'CTL', 'CONTROLE', 'TD']) else np.nan)

    # 1. Cinemática e Velocidade do Olhar
    dx = gaze_valido['Gaze_X'].diff().fillna(0)
    dy = gaze_valido['Gaze_Y'].diff().fillna(0)
    velocidade = np.sqrt(dx**2 + dy**2)
    vel_median = float(velocidade.median())
    vel_std = float(velocidade.std())

    # 2. BCEA Geral e por Emoção
    bcea_geral = calcular_bcea(df['Gaze_X'].values, df['Gaze_Y'].values)
    bcea_emo = {}
    if 'Emocao' in df.columns:
        for emo in ['feliz', 'neutro', 'raiva']:
            sub = df[df['Emocao'].astype(str).str.lower() == emo]
            bcea_emo[emo] = calcular_bcea(sub['Gaze_X'].values, sub['Gaze_Y'].values)
    else:
        bcea_emo = {'feliz': 0.0, 'neutro': 0.0, 'raiva': 0.0}

    # 3. Ratios de Reatividade Oculomotora (ROR)
    ror_raiva = bcea_emo['raiva'] / (bcea_emo['neutro'] + 1e-5)
    ror_feliz = bcea_emo['feliz'] / (bcea_emo['neutro'] + 1e-5)

    # 4. Proporções e Ratios de AOI
    tot = len(df)
    prop_olhos = df['AOI_Ambos_Olhos'].sum() / tot if 'AOI_Ambos_Olhos' in df.columns else 0.0
    prop_boca = df['AOI_Boca'].sum() / tot if 'AOI_Boca' in df.columns else 0.0
    prop_nariz = df['AOI_Nariz'].sum() / tot if 'AOI_Nariz' in df.columns else 0.0
    prop_rosto = df['AOI_Rosto'].sum() / tot if 'AOI_Rosto' in df.columns else 0.0
    
    prop_triangulo_rosto = (prop_olhos + prop_nariz) / (prop_rosto + 1e-5)

    features = {
        'ID_Sujeito': id_sujeito,
        'Target': target,
        'BCEA_Geral': bcea_geral,
        'BCEA_Feliz': bcea_emo['feliz'],
        'BCEA_Neutro': bcea_emo['neutro'],
        'BCEA_Raiva': bcea_emo['raiva'],
        'ROR_Raiva_Neutro': ror_raiva,
        'ROR_Feliz_Neutro': ror_feliz,
        'Prop_Olhos': prop_olhos,
        'Prop_Boca': prop_boca,
        'Prop_Triangulo_Rosto': prop_triangulo_rosto,
        'Gaze_Velocidade_Median': vel_median,
        'Gaze_Velocidade_Std': vel_std,
        'Gaze_X_Std': gaze_valido['Gaze_X'].std(),
        'Gaze_Y_Std': gaze_valido['Gaze_Y'].std()
    }
    return features

# Execução
lista_arquivos = glob.glob("/workspaces/EyeTracking/data/csv/*.csv")
dados = [res for f in lista_arquivos if (res := processar_arquivo_completo(f)) is not None]
df_mega = pd.DataFrame(dados)
df_mega.to_csv("/workspaces/EyeTracking/data/df_mega_bcea.csv", index=False)
print(f"Dataset atualizado com {len(df_mega)} sujeitos e {df_mega.shape[1]} atributos!")

Dataset atualizado com 41 sujeitos e 15 atributos!


In [9]:
import os
import glob
import numpy as np
import pandas as pd

def calcular_bcea(gx, gy):
    mask = ~np.isnan(gx) & ~np.isnan(gy) & (gx != 0) & (gy != 0)
    gx_clean, gy_clean = gx[mask], gy[mask]
    if len(gx_clean) < 3:
        return 0.0
    sx, sy = np.std(gx_clean, ddof=1), np.std(gy_clean, ddof=1)
    if sx == 0 or sy == 0:
        return 0.0
    rho = np.corrcoef(gx_clean, gy_clean)[0, 1]
    rho = 0.0 if np.isnan(rho) else np.clip(rho, -0.999, 0.999)
    return float(2 * np.pi * sx * sy * np.sqrt(max(0.0, 1 - rho**2)))

def processar_arquivo_estavel(caminho_csv):
    df = pd.read_csv(caminho_csv)
    
    # Tratamento numérico robusto
    cols_num = ['Gaze_X', 'Gaze_Y', 'Pupil_L', 'Pupil_R', 'AOI_Ambos_Olhos', 'AOI_Nariz', 'AOI_Boca', 'AOI_Rosto', 'AOI_Fora']
    for col in cols_num:
        if col in df.columns:
            if df[col].dtype == object or str(df[col].dtype).startswith('string'):
                df[col] = df[col].astype(str).str.replace(',', '.')
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    gaze_valido = df[(df['Gaze_X'] != 0) & (df['Gaze_Y'] != 0)].copy()
    if len(gaze_valido) < 50:
        return None
    
    nome_arquivo = os.path.basename(caminho_csv).replace('.csv', '')
    id_sujeito = nome_arquivo.split()[0]
    nome_upper = nome_arquivo.upper()
    target = 1 if 'TEA' in nome_upper else (0 if any(t in nome_upper for t in ['TDC', 'CTL', 'CONTROLE', 'TD']) else np.nan)

    # 1. Cinemática da Velocidade
    dx = gaze_valido['Gaze_X'].diff().fillna(0)
    dy = gaze_valido['Gaze_Y'].diff().fillna(0)
    velocidade = np.sqrt(dx**2 + dy**2)
    vel_median = float(velocidade.median())
    vel_std = float(velocidade.std())

    # 2. BCEAs por Emoção
    bcea_geral = calcular_bcea(df['Gaze_X'].values, df['Gaze_Y'].values)
    bcea_emo = {}
    if 'Emocao' in df.columns:
        for emo in ['feliz', 'neutro', 'raiva']:
            sub = df[df['Emocao'].astype(str).str.lower() == emo]
            bcea_emo[emo] = calcular_bcea(sub['Gaze_X'].values, sub['Gaze_Y'].values)
    else:
        bcea_emo = {'feliz': 0.0, 'neutro': 0.0, 'raiva': 0.0}

    # 3. Diferenças Logarítmicas de Reatividade (Substitui as divisões instáveis)
    log_bcea_raiva = np.log1p(bcea_emo['raiva'])
    log_bcea_feliz = np.log1p(bcea_emo['feliz'])
    log_bcea_neutro = np.log1p(bcea_emo['neutro'])

    diff_log_raiva_neutro = log_bcea_raiva - log_bcea_neutro
    diff_log_feliz_neutro = log_bcea_feliz - log_bcea_neutro

    # 4. Proporções de AOI
    tot = len(df)
    prop_olhos = df['AOI_Ambos_Olhos'].sum() / tot if 'AOI_Ambos_Olhos' in df.columns else 0.0
    prop_boca = df['AOI_Boca'].sum() / tot if 'AOI_Boca' in df.columns else 0.0
    prop_nariz = df['AOI_Nariz'].sum() / tot if 'AOI_Nariz' in df.columns else 0.0
    prop_rosto = df['AOI_Rosto'].sum() / tot if 'AOI_Rosto' in df.columns else 0.0

    features = {
        'ID_Sujeito': id_sujeito,
        'Target': target,
        'BCEA_Geral_Log': np.log1p(bcea_geral),
        'BCEA_Feliz_Log': log_bcea_feliz,
        'BCEA_Neutro_Log': log_bcea_neutro,
        'BCEA_Raiva_Log': log_bcea_raiva,
        'Diff_Log_Raiva_Neutro': diff_log_raiva_neutro,
        'Diff_Log_Feliz_Neutro': diff_log_feliz_neutro,
        'Prop_Olhos': prop_olhos,
        'Prop_Boca': prop_boca,
        'Prop_Nariz': prop_nariz,
        'Prop_Rosto': prop_rosto,
        'Gaze_Velocidade_Median': vel_median,
        'Gaze_Velocidade_Std': vel_std,
        'Gaze_X_Std': gaze_valido['Gaze_X'].std(),
        'Gaze_Y_Std': gaze_valido['Gaze_Y'].std()
    }
    return features

# Execução
lista_arquivos = glob.glob("/workspaces/EyeTracking/data/csv/*.csv")
dados = [res for f in lista_arquivos if (res := processar_arquivo_estavel(f)) is not None]
df_mega = pd.DataFrame(dados)
df_mega.to_csv("/workspaces/EyeTracking/data/df_mega_bcea.csv", index=False)
print(f"Dataset RECONSTRUÍDO com Sucesso: {len(df_mega)} sujeitos e {df_mega.shape[1]} colunas.")

Dataset RECONSTRUÍDO com Sucesso: 41 sujeitos e 16 colunas.


In [10]:
import os
import glob
import numpy as np
import pandas as pd

def calcular_bcea(gx, gy):
    mask = ~np.isnan(gx) & ~np.isnan(gy) & (gx != 0) & (gy != 0)
    gx_clean, gy_clean = gx[mask], gy[mask]
    if len(gx_clean) < 3:
        return 0.0, 0.0, 0.0
    sx, sy = np.std(gx_clean, ddof=1), np.std(gy_clean, ddof=1)
    if sx == 0 or sy == 0:
        return 0.0, 0.0, 0.0
    rho = np.corrcoef(gx_clean, gy_clean)[0, 1]
    rho = 0.0 if np.isnan(rho) else np.clip(rho, -0.999, 0.999)
    area = 2 * np.pi * sx * sy * np.sqrt(max(0.0, 1 - rho**2))
    anisotropia = sy / (sx + 1e-5)
    return float(area), float(sx), float(anisotropia)

def calcular_entropia_espacial(gx, gy, bins=10):
    mask = ~np.isnan(gx) & ~np.isnan(gy) & (gx != 0) & (gy != 0)
    gx_c, gy_c = gx[mask], gy[mask]
    if len(gx_c) < 10:
        return 0.0
    hist, _, _ = np.histogram2d(gx_c, gy_c, bins=bins)
    probs = hist.flatten() / np.sum(hist)
    probs = probs[probs > 0]
    return float(-np.sum(probs * np.log2(probs)))

def processar_arquivo_avancado(caminho_csv):
    df = pd.read_csv(caminho_csv)
    
    cols_num = ['Gaze_X', 'Gaze_Y', 'Pupil_L', 'Pupil_R', 'AOI_Ambos_Olhos', 'AOI_Nariz', 'AOI_Boca', 'AOI_Rosto', 'AOI_Fora']
    for col in cols_num:
        if col in df.columns:
            if df[col].dtype == object or str(df[col].dtype).startswith('string'):
                df[col] = df[col].astype(str).str.replace(',', '.')
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    gaze_valido = df[(df['Gaze_X'] != 0) & (df['Gaze_Y'] != 0)].copy()
    if len(gaze_valido) < 50:
        return None
    
    nome_arquivo = os.path.basename(caminho_csv).replace('.csv', '')
    id_sujeito = nome_arquivo.split()[0]
    nome_upper = nome_arquivo.upper()
    target = 1 if 'TEA' in nome_upper else (0 if any(t in nome_upper for t in ['TDC', 'CTL', 'CONTROLE', 'TD']) else np.nan)

    # 1. Cinemática e Dispersão Espacial
    dx = gaze_valido['Gaze_X'].diff().fillna(0)
    dy = gaze_valido['Gaze_Y'].diff().fillna(0)
    velocidade = np.sqrt(dx**2 + dy**2)
    
    bcea_geral, std_x, anisotropia = calcular_bcea(df['Gaze_X'].values, df['Gaze_Y'].values)
    entropia_2d = calcular_entropia_espacial(df['Gaze_X'].values, df['Gaze_Y'].values)

    # 2. BCEAs por Emoção (Log)
    bcea_emo = {}
    if 'Emocao' in df.columns:
        for emo in ['feliz', 'neutro', 'raiva']:
            sub = df[df['Emocao'].astype(str).str.lower() == emo]
            area, _, _ = calcular_bcea(sub['Gaze_X'].values, sub['Gaze_Y'].values)
            bcea_emo[emo] = np.log1p(area)
    else:
        bcea_emo = {'feliz': 0.0, 'neutro': 0.0, 'raiva': 0.0}

    # 3. Proporções de AOI
    tot = len(df)
    prop_olhos = df['AOI_Ambos_Olhos'].sum() / tot if 'AOI_Ambos_Olhos' in df.columns else 0.0
    prop_boca = df['AOI_Boca'].sum() / tot if 'AOI_Boca' in df.columns else 0.0
    prop_nariz = df['AOI_Nariz'].sum() / tot if 'AOI_Nariz' in df.columns else 0.0
    prop_rosto = df['AOI_Rosto'].sum() / tot if 'AOI_Rosto' in df.columns else 0.0
    prop_fora = df['AOI_Fora'].sum() / tot if 'AOI_Fora' in df.columns else 0.0

    features = {
        'ID_Sujeito': id_sujeito,
        'Target': target,
        'BCEA_Geral_Log': np.log1p(bcea_geral),
        'Anisotropia_Gaze': anisotropia,
        'Entropia_Espacial_2D': entropia_2d,
        'BCEA_Feliz_Log': bcea_emo['feliz'],
        'BCEA_Neutro_Log': bcea_emo['neutro'],
        'BCEA_Raiva_Log': bcea_emo['raiva'],
        'Diff_Log_Raiva_Neutro': bcea_emo['raiva'] - bcea_emo['neutro'],
        'Diff_Log_Feliz_Neutro': bcea_emo['feliz'] - bcea_emo['neutro'],
        'Prop_Olhos': prop_olhos,
        'Prop_Boca': prop_boca,
        'Prop_Nariz': prop_nariz,
        'Prop_Rosto': prop_rosto,
        'Prop_Fora': prop_fora,
        'Velocidade_Median': float(velocidade.median()),
        'Velocidade_Std': float(velocidade.std())
    }
    return features

# Execução
lista_arquivos = glob.glob("/workspaces/EyeTracking/data/csv/*.csv")
dados = [res for f in lista_arquivos if (res := processar_arquivo_avancado(f)) is not None]
df_mega = pd.DataFrame(dados)
df_mega.to_csv("/workspaces/EyeTracking/data/df_mega_bcea.csv", index=False)
print(f"Dataset recarregado com Sucesso: {len(df_mega)} sujeitos e {df_mega.shape[1]} atributos!")

Dataset recarregado com Sucesso: 41 sujeitos e 17 atributos!


In [11]:
import pandas as pd

# Inspecionar a estrutura interna de um arquivo bruto
df_exemplo = pd.read_csv("/workspaces/EyeTracking/data/csv/T28 TEA GRAVE.csv")

print("=" * 60)
print("INSPEÇÃO DA ESTRUTURA DE TRIALS NO ARQUIVO BRUTO")
print("=" * 60)
print("Colunas disponíveis no CSV:")
print(df_exemplo.columns.tolist())
print("\nVerificação de colunas de Estímulo / Bloco / Imagem:")
colunas_estimulo = [c for c in df_exemplo.columns if any(k in c.lower() for k in ['stim', 'image', 'trial', 'bloco', 'media', 'nome'])]
print(colunas_estimulo)

if colunas_estimulo:
    col = colunas_estimulo[0]
    print(f"\nValores únicos na coluna '{col}':")
    print(df_exemplo[col].value_counts().head(10))
print("=" * 60)

INSPEÇÃO DA ESTRUTURA DE TRIALS NO ARQUIVO BRUTO
Colunas disponíveis no CSV:
['Time_ms', 'Gaze_X', 'Gaze_Y', 'Pupil_L', 'Pupil_R', 'Face_ID', 'Tipo_Estimulo', 'Emocao', 'Fase_Estimulo', 'AOI_Olho_Esquerdo', 'AOI_Olho_Direito', 'AOI_Ambos_Olhos', 'AOI_Nariz', 'AOI_Boca', 'AOI_Triangulo', 'AOI_Rosto', 'AOI_Fora']

Verificação de colunas de Estímulo / Bloco / Imagem:
['Tipo_Estimulo', 'Fase_Estimulo']

Valores únicos na coluna 'Tipo_Estimulo':
Tipo_Estimulo
Humano     2475
Desenho     667
Nenhum      107
Name: count, dtype: int64


In [12]:
import os
import glob
import numpy as np
import pandas as pd

def calcular_bcea(gx, gy):
    mask = ~np.isnan(gx) & ~np.isnan(gy) & (gx != 0) & (gy != 0)
    gx_clean, gy_clean = gx[mask], gy[mask]
    if len(gx_clean) < 3:
        return 0.0, 0.0
    sx, sy = np.std(gx_clean, ddof=1), np.std(gy_clean, ddof=1)
    if sx == 0 or sy == 0:
        return 0.0, 0.0
    rho = np.corrcoef(gx_clean, gy_clean)[0, 1]
    rho = 0.0 if np.isnan(rho) else np.clip(rho, -0.999, 0.999)
    area = 2 * np.pi * sx * sy * np.sqrt(max(0.0, 1 - rho**2))
    anisotropia = sy / (sx + 1e-5)
    return float(area), float(anisotropia)

def processar_arquivo_por_estimulo(caminho_csv):
    df = pd.read_csv(caminho_csv)
    
    cols_num = ['Gaze_X', 'Gaze_Y', 'Pupil_L', 'Pupil_R', 'AOI_Ambos_Olhos', 'AOI_Nariz', 'AOI_Boca', 'AOI_Rosto', 'AOI_Fora']
    for col in cols_num:
        if col in df.columns:
            if df[col].dtype == object or str(df[col].dtype).startswith('string'):
                df[col] = df[col].astype(str).str.replace(',', '.')
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    nome_arquivo = os.path.basename(caminho_csv).replace('.csv', '')
    id_sujeito = nome_arquivo.split()[0]
    nome_upper = nome_arquivo.upper()
    target = 1 if 'TEA' in nome_upper else (0 if any(t in nome_upper for t in ['TDC', 'CTL', 'CONTROLE', 'TD']) else np.nan)

    if pd.isna(target):
        return []

    registros = []
    
    # Processa cada tipo de estímulo separadamente (Humano vs Desenho)
    for tipo_est in ['Humano', 'Desenho']:
        sub_df = df[df['Tipo_Estimulo'].astype(str).str.strip().str.capitalize() == tipo_est]
        gaze_valido = sub_df[(sub_df['Gaze_X'] != 0) & (sub_df['Gaze_Y'] != 0)]
        
        if len(gaze_valido) < 30:
            continue
            
        bcea_area, anisotropia = calcular_bcea(sub_df['Gaze_X'].values, sub_df['Gaze_Y'].values)
        
        dx = gaze_valido['Gaze_X'].diff().fillna(0)
        dy = gaze_valido['Gaze_Y'].diff().fillna(0)
        velocidade = np.sqrt(dx**2 + dy**2)
        
        tot = len(sub_df)
        prop_olhos = sub_df['AOI_Ambos_Olhos'].sum() / tot if 'AOI_Ambos_Olhos' in sub_df.columns else 0.0
        prop_boca = sub_df['AOI_Boca'].sum() / tot if 'AOI_Boca' in sub_df.columns else 0.0
        prop_nariz = sub_df['AOI_Nariz'].sum() / tot if 'AOI_Nariz' in sub_df.columns else 0.0
        prop_rosto = sub_df['AOI_Rosto'].sum() / tot if 'AOI_Rosto' in sub_df.columns else 0.0

        # Sub-análise por Emoção dentro do tipo de estímulo
        bcea_emo = {}
        for emo in ['feliz', 'neutro', 'raiva']:
            emo_df = sub_df[sub_df['Emocao'].astype(str).str.lower() == emo]
            area_e, _ = calcular_bcea(emo_df['Gaze_X'].values, emo_df['Gaze_Y'].values)
            bcea_emo[emo] = np.log1p(area_e)

        registros.append({
            'ID_Sujeito': id_sujeito,
            'Tipo_Estimulo': tipo_est,
            'Target': int(target),
            'BCEA_Log': np.log1p(bcea_area),
            'Anisotropia_Gaze': anisotropia,
            'Prop_Olhos': prop_olhos,
            'Prop_Boca': prop_boca,
            'Prop_Nariz': prop_nariz,
            'Prop_Rosto': prop_rosto,
            'Razao_Olhos_Boca': prop_olhos / (prop_boca + 1e-5),
            'Velocidade_Median': float(velocidade.median()),
            'Velocidade_Std': float(velocidade.std()),
            'BCEA_Feliz_Log': bcea_emo['feliz'],
            'BCEA_Neutro_Log': bcea_emo['neutro'],
            'BCEA_Raiva_Log': bcea_emo['raiva'],
            'Diff_Log_Raiva_Neutro': bcea_emo['raiva'] - bcea_emo['neutro']
        })

    return registros

# Execução
lista_arquivos = glob.glob("/workspaces/EyeTracking/data/csv/*.csv")
todos_registros = []
for f in lista_arquivos:
    todos_registros.extend(processar_arquivo_por_estimulo(f))

df_mega = pd.DataFrame(todos_registros)
df_mega.to_csv("/workspaces/EyeTracking/data/df_mega_bcea.csv", index=False)

print("=" * 70)
print(f"Dataset Granular Gerado com Sucesso!")
print(f"Total de Registros (Amostras): {len(df_mega)}")
print(f"Distribuição por Tipo de Estímulo:\n{df_mega['Tipo_Estimulo'].value_counts()}")
print("=" * 70)

Dataset Granular Gerado com Sucesso!
Total de Registros (Amostras): 82
Distribuição por Tipo de Estímulo:
Tipo_Estimulo
Humano     41
Desenho    41
Name: count, dtype: int64


In [13]:
import os
import glob
import numpy as np
import pandas as pd

def calcular_bcea(gx, gy):
    mask = ~np.isnan(gx) & ~np.isnan(gy) & (gx != 0) & (gy != 0)
    gx_clean, gy_clean = gx[mask], gy[mask]
    if len(gx_clean) < 3:
        return 0.0, 0.0
    sx, sy = np.std(gx_clean, ddof=1), np.std(gy_clean, ddof=1)
    if sx == 0 or sy == 0:
        return 0.0, 0.0
    rho = np.corrcoef(gx_clean, gy_clean)[0, 1]
    rho = 0.0 if np.isnan(rho) else np.clip(rho, -0.999, 0.999)
    area = 2 * np.pi * sx * sy * np.sqrt(max(0.0, 1 - rho**2))
    anisotropia = sy / (sx + 1e-5)
    return float(area), float(anisotropia)

def extrair_matriz_transicao(sub_df):
    states = []
    for _, row in sub_df.iterrows():
        if row.get('AOI_Olho_Esquerdo', 0) == 1 or row.get('AOI_Olho_Direito', 0) == 1 or row.get('AOI_Ambos_Olhos', 0) == 1:
            states.append('Olhos')
        elif row.get('AOI_Nariz', 0) == 1:
            states.append('Nariz')
        elif row.get('AOI_Boca', 0) == 1:
            states.append('Boca')
        else:
            states.append('Fora')
            
    states = np.array(states)
    if len(states) < 15:
        return {'Trans_Olhos_Fora': 0.0, 'Trans_Fora_Olhos': 0.0, 'Trans_Olhos_Olhos': 0.0, 'Trans_Fora_Fora': 0.0, 'Trans_Olhos_Boca': 0.0}
        
    unique_states = ['Olhos', 'Nariz', 'Boca', 'Fora']
    n_states = len(unique_states)
    state_to_idx = {s: i for i, s in enumerate(unique_states)}
    
    trans_matrix = np.zeros((n_states, n_states))
    for i in range(len(states) - 1):
        s1, s2 = states[i], states[i+1]
        if s1 in state_to_idx and s2 in state_to_idx:
            trans_matrix[state_to_idx[s1], state_to_idx[s2]] += 1
            
    row_sums = trans_matrix.sum(axis=1, keepdims=True)
    trans_prob = np.divide(trans_matrix, row_sums, out=np.zeros_like(trans_matrix), where=row_sums!=0)
    
    idx_olhos = state_to_idx['Olhos']
    idx_fora = state_to_idx['Fora']
    idx_boca = state_to_idx['Boca']
    
    return {
        'Trans_Olhos_Fora': float(trans_prob[idx_olhos, idx_fora]),
        'Trans_Fora_Olhos': float(trans_prob[idx_fora, idx_olhos]),
        'Trans_Olhos_Olhos': float(trans_prob[idx_olhos, idx_olhos]),
        'Trans_Fora_Fora': float(trans_prob[idx_fora, idx_fora]),
        'Trans_Olhos_Boca': float(trans_prob[idx_olhos, idx_boca])
    }

def processar_arquivo_por_estimulo(caminho_csv):
    df = pd.read_csv(caminho_csv)
    
    cols_num = ['Gaze_X', 'Gaze_Y', 'Pupil_L', 'Pupil_R', 'AOI_Olho_Esquerdo', 'AOI_Olho_Direito', 
                'AOI_Ambos_Olhos', 'AOI_Nariz', 'AOI_Boca', 'AOI_Rosto', 'AOI_Fora']
    for col in cols_num:
        if col in df.columns:
            if df[col].dtype == object or str(df[col].dtype).startswith('string'):
                df[col] = df[col].astype(str).str.replace(',', '.')
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    nome_arquivo = os.path.basename(caminho_csv).replace('.csv', '')
    id_sujeito = nome_arquivo.split()[0]
    nome_upper = nome_arquivo.upper()
    target = 1 if 'TEA' in nome_upper else (0 if any(t in nome_upper for t in ['TDC', 'CTL', 'CONTROLE', 'TD']) else np.nan)

    if pd.isna(target):
        return []

    registros = []
    
    for tipo_est in ['Humano', 'Desenho']:
        sub_df = df[df['Tipo_Estimulo'].astype(str).str.strip().str.capitalize() == tipo_est]
        gaze_valido = sub_df[(sub_df['Gaze_X'] != 0) & (sub_df['Gaze_Y'] != 0)]
        
        if len(gaze_valido) < 30:
            continue
            
        bcea_area, anisotropia = calcular_bcea(sub_df['Gaze_X'].values, sub_df['Gaze_Y'].values)
        trans_feats = extrair_matriz_transicao(sub_df)
        
        dx = gaze_valido['Gaze_X'].diff().fillna(0)
        dy = gaze_valido['Gaze_Y'].diff().fillna(0)
        velocidade = np.sqrt(dx**2 + dy**2)
        
        tot = len(sub_df)
        prop_olhos = sub_df['AOI_Ambos_Olhos'].sum() / tot if 'AOI_Ambos_Olhos' in sub_df.columns else 0.0
        prop_boca = sub_df['AOI_Boca'].sum() / tot if 'AOI_Boca' in sub_df.columns else 0.0
        prop_rosto = sub_df['AOI_Rosto'].sum() / tot if 'AOI_Rosto' in sub_df.columns else 0.0

        registros.append({
            'ID_Sujeito': id_sujeito,
            'Tipo_Estimulo': tipo_est,
            'Target': int(target),
            'BCEA_Log': np.log1p(bcea_area),
            'Anisotropia_Gaze': anisotropia,
            'Prop_Olhos': prop_olhos,
            'Prop_Boca': prop_boca,
            'Prop_Rosto': prop_rosto,
            'Velocidade_Median': float(velocidade.median()),
            'Velocidade_Std': float(velocidade.std()),
            **trans_feats
        })

    return registros

# Execução
lista_arquivos = glob.glob("/workspaces/EyeTracking/data/csv/*.csv")
todos_registros = []
for f in lista_arquivos:
    todos_registros.extend(processar_arquivo_por_estimulo(f))

df_mega = pd.DataFrame(todos_registros)
df_mega.to_csv("/workspaces/EyeTracking/data/df_mega_bcea.csv", index=False)

print("=" * 70)
print(f"Dataset Granular com Transições Gerado com Sucesso!")
print(f"Total de Registros: {len(df_mega)} | Total de Features: {df_mega.shape[1]}")
print("=" * 70)

Dataset Granular com Transições Gerado com Sucesso!
Total de Registros: 82 | Total de Features: 15


In [14]:
import os
import glob
import numpy as np
import pandas as pd

def calcular_bcea(gx, gy):
    mask = ~np.isnan(gx) & ~np.isnan(gy) & (gx != 0) & (gy != 0)
    gx_clean, gy_clean = gx[mask], gy[mask]
    if len(gx_clean) < 3:
        return 0.0, 0.0
    sx, sy = np.std(gx_clean, ddof=1), np.std(gy_clean, ddof=1)
    if sx == 0 or sy == 0:
        return 0.0, 0.0
    rho = np.corrcoef(gx_clean, gy_clean)[0, 1]
    rho = 0.0 if np.isnan(rho) else np.clip(rho, -0.999, 0.999)
    area = 2 * np.pi * sx * sy * np.sqrt(max(0.0, 1 - rho**2))
    anisotropia = sy / (sx + 1e-5)
    return float(area), float(anisotropia)

def processar_por_condicao_especifica(caminho_csv):
    df = pd.read_csv(caminho_csv)
    
    cols_num = ['Gaze_X', 'Gaze_Y', 'Pupil_L', 'Pupil_R', 'AOI_Olho_Esquerdo', 'AOI_Olho_Direito', 
                'AOI_Ambos_Olhos', 'AOI_Nariz', 'AOI_Boca', 'AOI_Rosto', 'AOI_Fora']
    for col in cols_num:
        if col in df.columns:
            if df[col].dtype == object or str(df[col].dtype).startswith('string'):
                df[col] = df[col].astype(str).str.replace(',', '.')
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    nome_arquivo = os.path.basename(caminho_csv).replace('.csv', '')
    id_sujeito = nome_arquivo.split()[0]
    nome_upper = nome_arquivo.upper()
    target = 1 if 'TEA' in nome_upper else (0 if any(t in nome_upper for t in ['TDC', 'CTL', 'CONTROLE', 'TD']) else np.nan)

    if pd.isna(target):
        return []

    registros = []
    
    # Cruzamento exato: Tipo de Estímulo x Emoção
    for tipo_est in ['Humano', 'Desenho']:
        sub_tipo = df[df['Tipo_Estimulo'].astype(str).str.strip().str.capitalize() == tipo_est]
        
        for emo in ['feliz', 'neutro', 'raiva']:
            sub_emo = sub_tipo[sub_tipo['Emocao'].astype(str).str.strip().str.lower() == emo]
            gaze_valido = sub_emo[(sub_emo['Gaze_X'] != 0) & (sub_emo['Gaze_Y'] != 0)]
            
            # Se houver dados suficientes para essa condição específica (~3 segundos por face * nº de faces)
            if len(gaze_valido) < 20:
                continue
                
            bcea_area, anisotropia = calcular_bcea(sub_emo['Gaze_X'].values, sub_emo['Gaze_Y'].values)
            
            dx = gaze_valido['Gaze_X'].diff().fillna(0)
            dy = gaze_valido['Gaze_Y'].diff().fillna(0)
            velocidade = np.sqrt(dx**2 + dy**2)
            
            tot = len(sub_emo) if len(sub_emo) > 0 else 1
            prop_olhos = sub_emo['AOI_Ambos_Olhos'].sum() / tot if 'AOI_Ambos_Olhos' in sub_emo.columns else 0.0
            prop_boca = sub_emo['AOI_Boca'].sum() / tot if 'AOI_Boca' in sub_emo.columns else 0.0
            prop_rosto = sub_emo['AOI_Rosto'].sum() / tot if 'AOI_Rosto' in sub_emo.columns else 0.0
            prop_fora = sub_emo['AOI_Fora'].sum() / tot if 'AOI_Fora' in sub_emo.columns else 0.0

            registros.append({
                'ID_Sujeito': id_sujeito,
                'Condicao': f"{tipo_est}_{emo.capitalize()}",
                'Target': int(target),
                'BCEA_Log': np.log1p(bcea_area),
                'Anisotropia_Gaze': anisotropia,
                'Prop_Olhos': prop_olhos,
                'Prop_Boca': prop_boca,
                'Prop_Rosto': prop_rosto,
                'Prop_Fora': prop_fora,
                'Velocidade_Median': float(velocidade.median()),
                'Velocidade_Std': float(velocidade.std())
            })

    return registros

# Execução
lista_arquivos = glob.glob("/workspaces/EyeTracking/data/csv/*.csv")
todos_registros = []
for f in lista_arquivos:
    todos_registros.extend(processar_por_condicao_especifica(f))

df_mega = pd.DataFrame(todos_registros)
df_mega.to_csv("/workspaces/EyeTracking/data/df_mega_bcea.csv", index=False)

print("=" * 70)
print(f"Dataset por Condição Específica Gerado com Sucesso!")
print(f"Total de Amostras: {len(df_mega)}")
print(f"Contagem por Condição:\n{df_mega['Condicao'].value_counts()}")
print("=" * 70)

Dataset por Condição Específica Gerado com Sucesso!
Total de Amostras: 246
Contagem por Condição:
Condicao
Humano_Feliz      41
Humano_Neutro     41
Humano_Raiva      41
Desenho_Feliz     41
Desenho_Neutro    41
Desenho_Raiva     41
Name: count, dtype: int64


In [15]:
import os
import glob
import numpy as np
import pandas as pd

def calcular_bcea(gx, gy):
    mask = ~np.isnan(gx) & ~np.isnan(gy) & (gx != 0) & (gy != 0)
    gx_clean, gy_clean = gx[mask], gy[mask]
    if len(gx_clean) < 3:
        return 0.0, 0.0
    sx, sy = np.std(gx_clean, ddof=1), np.std(gy_clean, ddof=1)
    if sx == 0 or sy == 0:
        return 0.0, 0.0
    rho = np.corrcoef(gx_clean, gy_clean)[0, 1]
    rho = 0.0 if np.isnan(rho) else np.clip(rho, -0.999, 0.999)
    area = 2 * np.pi * sx * sy * np.sqrt(max(0.0, 1 - rho**2))
    anisotropia = sy / (sx + 1e-5)
    return float(area), float(anisotropia)

def processar_perfil_sujeito(caminho_csv):
    df = pd.read_csv(caminho_csv)
    
    cols_num = ['Gaze_X', 'Gaze_Y', 'Pupil_L', 'Pupil_R', 'AOI_Olho_Esquerdo', 'AOI_Olho_Direito', 
                'AOI_Ambos_Olhos', 'AOI_Nariz', 'AOI_Boca', 'AOI_Rosto', 'AOI_Fora']
    for col in cols_num:
        if col in df.columns:
            if df[col].dtype == object or str(df[col].dtype).startswith('string'):
                df[col] = df[col].astype(str).str.replace(',', '.')
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    nome_arquivo = os.path.basename(caminho_csv).replace('.csv', '')
    id_sujeito = nome_arquivo.split()[0]
    nome_upper = nome_arquivo.upper()
    target = 1 if 'TEA' in nome_upper else (0 if any(t in nome_upper for t in ['TDC', 'CTL', 'CONTROLE', 'TD']) else np.nan)

    if pd.isna(target):
        return None

    features_sujeito = {'ID_Sujeito': id_sujeito, 'Target': int(target)}
    
    # Extrai métricas para cada combinação exata
    for tipo_est in ['Humano', 'Desenho']:
        sub_tipo = df[df['Tipo_Estimulo'].astype(str).str.strip().str.capitalize() == tipo_est]
        
        for emo in ['feliz', 'neutro', 'raiva']:
            sub_emo = sub_tipo[sub_tipo['Emocao'].astype(str).str.strip().str.lower() == emo]
            gaze_valido = sub_emo[(sub_emo['Gaze_X'] != 0) & (sub_emo['Gaze_Y'] != 0)]
            
            if len(gaze_valido) < 10:
                bcea_val, aniso_val, p_olhos, p_boca = 0.0, 0.0, 0.0, 0.0
            else:
                bcea_val, aniso_val = calcular_bcea(sub_emo['Gaze_X'].values, sub_emo['Gaze_Y'].values)
                tot = len(sub_emo) if len(sub_emo) > 0 else 1
                p_olhos = sub_emo['AOI_Ambos_Olhos'].sum() / tot if 'AOI_Ambos_Olhos' in sub_emo.columns else 0.0
                p_boca = sub_emo['AOI_Boca'].sum() / tot if 'AOI_Boca' in sub_emo.columns else 0.0
            
            prefixo = f"{tipo_est}_{emo.capitalize()}"
            features_sujeito[f"{prefixo}_BCEA"] = np.log1p(bcea_val)
            features_sujeito[f"{prefixo}_Olhos"] = p_olhos
            features_sujeito[f"{prefixo}_Boca"] = p_boca

    # Criação de Deltas (Reatividade: Emoção vs Neutro)
    for tipo in ['Humano', 'Desenho']:
        neutro_olhos = features_sujeito.get(f"{tipo}_Neutro_Olhos", 0)
        neutro_bcea = features_sujeito.get(f"{tipo}_Neutro_BCEA", 0)
        
        for emo in ['Feliz', 'Raiva']:
            features_sujeito[f"Delta_{tipo}_{emo}_vs_Neutro_Olhos"] = features_sujeito.get(f"{tipo}_{emo}_Olhos", 0) - neutro_olhos
            features_sujeito[f"Delta_{tipo}_{emo}_vs_Neutro_BCEA"] = features_sujeito.get(f"{tipo}_{emo}_BCEA", 0) - neutro_bcea

    return features_sujeito

# Execução
lista_arquivos = glob.glob("/workspaces/EyeTracking/data/csv/*.csv")
dados_sujeitos = [res for f in lista_arquivos if (res := processar_perfil_sujeito(f)) is not None]

df_mega = pd.DataFrame(dados_sujeitos)
df_mega.to_csv("/workspaces/EyeTracking/data/df_mega_bcea.csv", index=False)

print("=" * 70)
print(f"Dataset de Perfil por Sujeito Gerado com Sucesso!")
print(f"Total de Sujeitos: {len(df_mega)} | Total de Atributos: {df_mega.shape[1]}")
print("=" * 70)

Dataset de Perfil por Sujeito Gerado com Sucesso!
Total de Sujeitos: 41 | Total de Atributos: 28
